# ML-based Labeling

This notebook uses the manually verified/high-quality labels from `Final_Dataset_Label.csv` to train a Machine Learning model.
It then uses this model to predict labels for the larger dataset (`merged_supply_data_01.csv`).

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## 1. Load Training Data
Loading the Ground Truth dataset. **Note**: The file appears to have no header, so we supply column names.

In [ ]:
train_path = 'data/outputs/Final_Dataset_Label.csv'

col_names = ['source', 'raw_text', 'language', 'label_gemini', 'label_gpt', 'label_final', 'year']

df_train = pd.read_csv(train_path, names=col_names, header=None)

print(f"Training data shape: {df_train.shape}")
print(df_train.head())

In [ ]:
df_train = df_train.dropna(subset=['raw_text', 'label_final'])
df_train['raw_text'] = df_train['raw_text'].astype(str).str.lower()
df_train['label_final'] = df_train['label_final'].astype(str).str.lower()

print(f"Cleaned training data shape: {df_train.shape}")
print("Labels distribution:")
print(df_train['label_final'].value_counts())

## 2. Train Model
We use a **TF-IDF Vectorizer** to convert text to features and a **LinearSVC** for classification.

In [ ]:
X = df_train['raw_text']
y = df_train['label_final']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', LinearSVC(class_weight='balanced', random_state=42))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
pipeline.fit(X, y)
print("Model retrained on full dataset.")

## 3. Label the Rest of the Data
Loading the target dataset (`merged_supply_data_01.csv`) and predicting labels.

In [ ]:
target_path = 'data/outputs/merged_supply_data_01.csv'
output_labeled_path = 'data/outputs/merged_supply_data_ML_labeled.csv'

if hasattr(pd, 'read_csv'):
    df_target = pd.read_csv(target_path)
else:
    raise ImportError("Pandas not loaded properly")

print(f"Target data shape: {df_target.shape}")
print(df_target.head())

In [ ]:
df_target['raw_text'] = df_target['raw_text'].fillna('').astype(str).str.lower()

print("Predicting labels for target data...")
predicted_labels = pipeline.predict(df_target['raw_text'])

df_target['label_ml'] = predicted_labels

df_target['label_final'] = df_target['label_ml']

print("Prediction complete.")
print(df_target[['raw_text', 'label_final']].head())

In [ ]:
df_target.to_csv(output_labeled_path, index=False)
print(f"Saved ML-labeled data to {output_labeled_path}")